In [1]:
import os
import numpy as np
import pandas as pd

# 1. Define the requested target path
folder_path = r"C:\Users\sunda\Desktop\Retail _Banking _Data\Data"

# Ensure the local directory structure exists
os.makedirs(folder_path, exist_ok=True)
file_path = os.path.join(folder_path, "retail_banking_data.csv")

# 2. Set seed for mathematical reproducibility
np.random.seed(42)
n_rows = 10000

# 3. Authentic name pairing pools
first_names = ['James', 'Mary', 'John', 'Patricia', 'Robert', 'Jennifer', 'Michael', 'Linda', 'William', 'Elizabeth',
               'David', 'Barbara', 'Richard', 'Susan', 'Joseph', 'Jessica', 'Thomas', 'Sarah', 'Charles', 'Karen',
               'Christopher', 'Nancy', 'Daniel', 'Lisa', 'Matthew', 'Betty', 'Anthony', 'Margaret', 'Mark', 'Sandra']

last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Rodriguez', 'Martinez',
              'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson', 'Thomas', 'Taylor', 'Moore', 'Jackson', 'Martin']

# 4. Generate Demographics
customer_ids = [f"CUST-{i:05d}" for i in range(1, n_rows + 1)]
names = [f"{np.random.choice(first_names)} {np.random.choice(last_names)}" for _ in range(n_rows)]
ages = np.random.randint(18, 86, size=n_rows)
genders = np.random.choice(['Male', 'Female', 'Non-Binary'], size=n_rows, p=[0.49, 0.49, 0.02])
incomes = np.random.lognormal(mean=10.8, sigma=0.5, size=n_rows).astype(int)
credit_scores = np.clip(np.random.normal(680, 70, size=n_rows).astype(int), 300, 850)

# 5. Generate Account Properties & Balances
account_types = np.random.choice(['Standard Checking', 'Premium Checking', 'Savings', 'Money Market'], size=n_rows, p=[0.4, 0.2, 0.3, 0.1])
balances = np.where(account_types == 'Premium Checking', np.random.uniform(5000, 75000, size=n_rows),
                    np.where(account_types == 'Money Market', np.random.uniform(10000, 150000, size=n_rows),
                             np.random.uniform(50, 15000, size=n_rows))).astype(float).round(2)

# 6. Generate Product Ownership (1 = Yes, 0 = No)
has_credit_card = np.random.choice([1, 0], size=n_rows, p=[0.65, 0.35])
has_mortgage = np.random.choice([1, 0], size=n_rows, p=[0.15, 0.85])
has_personal_loan = np.random.choice([1, 0], size=n_rows, p=[0.20, 0.80])

# 7. Generate Digital Engagement Metrics
digital_engagement = np.random.choice(['High', 'Medium', 'Low', 'None'], size=n_rows, p=[0.35, 0.40, 0.18, 0.07])
online_trans_pct = np.where(digital_engagement == 'High', np.random.uniform(70, 100, n_rows),
                             np.where(digital_engagement == 'Medium', np.random.uniform(30, 70, n_rows),
                                      np.where(digital_engagement == 'Low', np.random.uniform(5, 30, n_rows), 0))).round(1)

# 8. Customer Sentiment & Custom Churn Risk Logic
complaints_filed = np.random.choice([1, 0], size=n_rows, p=[0.04, 0.96])

# Dynamic logic: Lower balances, lower credit scores, and zero digital usage compound the probability of Churn
churn_prob = (1.0 / (1.0 + np.exp(-(0.01 * (600 - credit_scores) + 0.0001 * (10000 - balances) + (digital_engagement == 'None') * 1.5))))
churn_status = np.where(churn_prob > 0.65, 1, 0)

# 9. Structure into final DataFrame
df = pd.DataFrame({
    'CustomerID': customer_ids,
    'CustomerName': names,
    'Age': ages,
    'Gender': genders,
    'AnnualIncome': incomes,
    'CreditScore': credit_scores,
    'AccountType': account_types,
    'AccountBalance': balances,
    'HasCreditCard': has_credit_card,
    'HasMortgage': has_mortgage,
    'HasPersonalLoan': has_personal_loan,
    'DigitalEngagementLevel': digital_engagement,
    'OnlineTransactionsPct': online_trans_pct,
    'HasComplained': complaints_filed,
    'ChurnStatus': churn_status
})

# 10. Write directly to disk
df.to_csv(file_path, index=False)

print("Dataset successfully compiled!")
print(f"File Path: {file_path}")
print(f"Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")


Dataset successfully compiled!
File Path: C:\Users\sunda\Desktop\Retail _Banking _Data\Data\retail_banking_data.csv
Dimensions: 10000 rows, 15 columns


In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Update the folder path to your Power BI directory
data_path = r"C:\Users\sunda\Desktop\Retail _Banking _Data\Data\retail_banking_data.csv"
power_bi_folder = r"C:\Users\sunda\Desktop\Retail _Banking _Data\Power BI"

# Create the Power BI folder if it does not exist yet
os.makedirs(power_bi_folder, exist_ok=True)

# Load the dataset
if not os.path.exists(data_path):
    print(f"Error: The file could not be found at {data_path}")
else:
    df = pd.read_csv(data_path)
    
    # Set up global styling
    sns.set_theme(style="whitegrid")
    plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 14})
    
    print("🚀 Exporting charts to Power BI folder...")

    # --- CHART 1: Churn Risk Watchlist ---
    plt.figure(figsize=(6, 4))
    churn_counts = df['ChurnStatus'].map({1: 'Churned', 0: 'Retained'}).value_counts(normalize=True) * 100
    sns.barplot(x=churn_counts.index, y=churn_counts.values, palette=['#4C72B0', '#C44E52'])
    plt.title('Baseline Customer Attrition Distribution')
    plt.ylabel('Percentage of Customer Base (%)')
    for i, v in enumerate(churn_counts.values):
        plt.text(i, v + 1, f"{v:.1f}%", ha='center', fontweight='bold')
    plt.tight_layout()
    # Save directly to the Power BI folder
    plt.savefig(os.path.join(power_bi_folder, '1_customer_churn_rate.png'), dpi=300)
    plt.close()

    # --- CHART 2: Cross-Selling Gaps ---
    plt.figure(figsize=(7, 4))
    products = ['HasCreditCard', 'HasMortgage', 'HasPersonalLoan']
    product_labels = ['Credit Card', 'Mortgage', 'Personal Loan']
    penetration = [df[p].mean() * 100 for p in products]
    sns.barplot(x=product_labels, y=penetration, palette='Blues_r')
    plt.title('Product Penetration Across Portfolio')
    plt.ylabel('Penetration Rate (%)')
    plt.ylim(0, 100)
    for i, v in enumerate(penetration):
        plt.text(i, v + 2, f"{v:.1f}%", ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(power_bi_folder, '2_product_cross_sell_penetration.png'), dpi=300)
    plt.close()

    # --- CHART 3: Digital Engagement vs Churn Risk ---
    plt.figure(figsize=(7, 4))
    digital_order = ['None', 'Low', 'Medium', 'High']
    chart_data = df.groupby('DigitalEngagementLevel')['ChurnStatus'].mean().reindex(digital_order) * 100
    sns.lineplot(x=chart_data.index, y=chart_data.values, marker='o', color='#55A868', linewidth=2.5, markersize=8)
    plt.title('Impact of Digital Adoption on Churn Risk')
    plt.ylabel('Observed Churn Rate (%)')
    for i, v in enumerate(chart_data.values):
        plt.text(i, v + 1, f"{v:.1f}%", ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(power_bi_folder, '3_digital_engagement_vs_churn.png'), dpi=300)
    plt.close()

    # --- CHART 4: Credit Risk Portfolio ---
    plt.figure(figsize=(8, 4))
    def assign_credit_tier(score):
        if score >= 720: return 'Superprime'
        elif score >= 660: return 'Prime'
        elif score >= 620: return 'Near-Prime'
        elif score >= 580: return 'Subprime'
        else: return 'Deep Subprime'
    df['CreditTier'] = df['CreditScore'].apply(assign_credit_tier)
    tier_order = ['Superprime', 'Prime', 'Near-Prime', 'Subprime', 'Deep Subprime']
    tier_counts = df['CreditTier'].value_counts().reindex(tier_order)
    sns.barplot(x=tier_counts.index, y=tier_counts.values, palette='Oranges_r')
    plt.title('Credit Risk Distribution Analysis')
    plt.ylabel('Customer Count')
    for i, v in enumerate(tier_counts.values):
        plt.text(i, v + 40, f"{v}", ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(power_bi_folder, '4_credit_risk_distribution.png'), dpi=300)
    plt.close()

    # --- CHART 5: Wealth Management Segmentation ---
    plt.figure(figsize=(5, 5))
    def segment_wealth(row):
        if row['AccountBalance'] >= 100000 or row['AnnualIncome'] >= 150000:
            return 'Private Banking / HNW'
        elif row['AccountBalance'] >= 25000 or row['AnnualIncome'] >= 80000:
            return 'Mass Affluent'
        else:
            return 'Retail Core'
    df['CustomerSegment'] = df.apply(segment_wealth, axis=1)
    segment_counts = df['CustomerSegment'].value_counts()
    plt.pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%', startangle=140, 
            colors=['#DD8452', '#4C72B0', '#937860'], wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    plt.title('Portfolio Breakdown by Affluence Category')
    plt.tight_layout()
    plt.savefig(os.path.join(power_bi_folder, '5_wealth_segmentation_pie.png'), dpi=300)
    plt.close()

    print("🏁 Done! All 5 charts are now saved as static images in your Power BI folder.")


🚀 Exporting charts to Power BI folder...


C:\Users\sunda\AppData\Local\Temp\ipykernel_7716\3543272727.py:29: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=churn_counts.index, y=churn_counts.values, palette=['#4C72B0', '#C44E52'])
C:\Users\sunda\AppData\Local\Temp\ipykernel_7716\3543272727.py:44: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=product_labels, y=penetration, palette='Blues_r')
C:\Users\sunda\AppData\Local\Temp\ipykernel_7716\3543272727.py:63: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
C:\Users\sunda\AppData\Local\Temp\ipykernel_7716\3543272727.py:78: FutureWarning: 

Passing `palette` without assigning `hue

🏁 Done! All 5 charts are now saved as static images in your Power BI folder.


In [ ]:
#BUSINESS PROBLEM 1: Customer Churn Mitigation (Proactive Retention)

In [6]:
    overall_churn_rate = df['ChurnStatus'].mean() * 100
    print(f"Overall Institution Churn Rate: {overall_churn_rate:.2f}%")
    
    high_risk_watchlist = df[
        (df['ChurnStatus'] == 1) & 
        ((df['DigitalEngagementLevel'] == 'None') | (df['DigitalEngagementLevel'] == 'Low'))
    ].sort_values(by='AccountBalance', ascending=False)
    
    print(f"Total High-Risk, Low-Engagement Customers Flagged: {len(high_risk_watchlist)}")
    print("\nTop 5 High-Value Accounts on Immediate Watchlist:")
    print(high_risk_watchlist[['CustomerID', 'CustomerName', 'AccountBalance', 'CreditScore', 'HasComplained']].head(5))
    print("\n[Displaying Visualisation 1 Below]")

Overall Institution Churn Rate: 8.55%
Total High-Risk, Low-Engagement Customers Flagged: 101

Top 5 High-Value Accounts on Immediate Watchlist:
      CustomerID       CustomerName  AccountBalance  CreditScore  \
1092  CUST-01093     Patricia Smith        12277.09          475   
8550  CUST-08551   Michael Williams        12081.73          501   
3541  CUST-03542        Betty Brown        10665.96          467   
809   CUST-00810        Mary Miller         9584.17          510   
4299  CUST-04300  Jennifer Martinez         9206.82          533   

      HasComplained  
1092              0  
8550              0  
3541              0  
809               0  
4299              0  

[Displaying Visualisation 1 Below]


In [ ]:
#BUSINESS PROBLEM 2.Cross-Selling & Wallet Share OptimizationWhy it matters: Multi-product customers are significantl

In [13]:
median_balance = df['AccountBalance'].median()

cross_sell_targets = df[
    (df['AccountBalance'] > median_balance) &
    (df['HasCreditCard'] == 0) &
    (df['HasMortgage'] == 0) &
    (df['HasPersonalLoan'] == 0)
].sort_values(by='AnnualIncome', ascending=False)

print(f"Total Core Accounts Eligible for Multi-Product Campaigns: {len(cross_sell_targets)}")
median_balance = df['AccountBalance'].median()

cross_sell_targets = df[
    (df['AccountBalance'] > median_balance) &
    (df['HasCreditCard'] == 0) &
    (df['HasMortgage'] == 0) &
    (df['HasPersonalLoan'] == 0)
].sort_values(by='AnnualIncome', ascending=False)

print(f"Total High-Potential Cross-Sell Targets Found: {len(cross_sell_targets)}")
print("\n--- TOP 5 CROSS-SELL MARKETING LEADS ---")
print(cross_sell_targets[['CustomerID', 'CustomerName', 'AnnualIncome', 'AccountBalance', 'CreditScore']].head(5))

Total Core Accounts Eligible for Multi-Product Campaigns: 1201
Total High-Potential Cross-Sell Targets Found: 1201

--- TOP 5 CROSS-SELL MARKETING LEADS ---
      CustomerID      CustomerName  AnnualIncome  AccountBalance  CreditScore
7872  CUST-07873     Karen Jackson        380684        10691.54          676
5487  CUST-05488    Jessica Thomas        254347        10595.71          683
332   CUST-00333  Michael Williams        249061        14610.93          708
9696  CUST-09697      Mary Jackson        177774        13000.59          561
9285  CUST-09286       Linda Moore        165963       140969.27          721


In [ ]:
#BUSINESS PROBLEM 3. Digital Transformation & Channel Migration Tracking

In [14]:
# Aggregate balances and churn rates across different digital engagement segments
digital_metrics = df.groupby('DigitalEngagementLevel').agg(
    Customer_Count=('CustomerID', 'count'),
    Average_Balance=('AccountBalance', 'mean'),
    Average_Online_Trans_Pct=('OnlineTransactionsPct', 'mean'),
    Churn_Rate=('ChurnStatus', 'mean')
).reset_index()

# Convert Churn Status average to a clean percentage format
digital_metrics['Churn_Rate'] = (digital_metrics['Churn_Rate'] * 100).round(2)
digital_metrics['Average_Balance'] = digital_metrics['Average_Balance'].round(2)

print("--- DIGITAL CHANNEL ENGAGEMENT METRICS ---")
print(digital_metrics.to_string(index=False))

--- DIGITAL CHANNEL ENGAGEMENT METRICS ---
DigitalEngagementLevel  Customer_Count  Average_Balance  Average_Online_Trans_Pct  Churn_Rate
                  High            3469         20766.27                 84.913750        5.45
                   Low            1700         21827.32                 17.871706        5.94
                Medium            4107         21314.51                 50.432067        5.28


In [ ]:
#BUSINESS PROBLEM 4. Risk Tiering & Quality of Credit Portfolio